## setup

In [140]:
# import stuff
from pathlib import Path
import pandas as pd
import numpy as np

In [141]:
# load data from folder
folder = Path('../data')

# filtered dataset with mortgage rates
sold = pd.read_csv(folder / 'sold_with_rates.csv', low_memory = False)

# null count summary as reference for cleaning
sold_null_summary = pd.read_csv(folder / 'sold_null_summary.csv', index_col = 0)

In [142]:
# ensure datasets loaded properly
print('Sold dataset shape:', sold.shape)
sold.head()

Sold dataset shape: (448253, 84)


,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,BuyerAgentAOR,ListAgentAOR,OriginatingSystemName,OriginatingSystemSubName,year_month,rate_30yr_fixed
0,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,...,94401,6472.0,NaN,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425
1,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,...,91950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425
2,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,...,91950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425
3,NaN,True,NaN,NaN,False,2100000.0,1067652762,jenniferarielkennedy@gmail.com,2024-01-02,2100000.0,...,93401,0.0,11219.0,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425
4,Tile,True,NaN,NaN,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,...,95437,NaN,74487.6,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425


In [143]:
sold_null_summary.head()

,null ct,null pct
Flooring,160819,35.876837
ViewYN,38504,8.589792
WaterfrontYN,447966,99.935974
BasementYN,439466,98.039723
PoolPrivateYN,38210,8.524204


## data cleaning & prep
- convert date fields to datetime format
- remove unnecessary/redundant cols
- handle missing values appropriately
- ensure numeric fields are properly typed
- remove/flag invalid numeric values

In [144]:
# convert date columns to datetime format
date_cols = ['CloseDate',
             'PurchaseContractDate',
             'ListingContractDate',
             'ContractStatusChangeDate']
sold[date_cols] = sold[date_cols].apply(pd.to_datetime, errors = 'coerce')

# check that changes have been made
sold[date_cols].dtypes

CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

In [145]:
# check cols w >90% nulls
flag_over_90 = sold_null_summary[sold_null_summary['null pct'] > 90].index.tolist()
flag_over_90.sort()
print(len(flag_over_90))
flag_over_90

15


['AboveGradeFinishedArea',
 'BasementYN',
 'BelowGradeFinishedArea',
 'BuilderName',
 'BuildingAreaTotal',
 'BusinessType',
 'CoBuyerAgentFirstName',
 'CoveredSpaces',
 'ElementarySchoolDistrict',
 'FireplacesTotal',
 'LotSizeDimensions',
 'MiddleOrJuniorSchoolDistrict',
 'TaxAnnualAmount',
 'TaxYear',
 'WaterfrontYN']

From the ``Real_Estate_Primer.pdf``, recall that our key data fields are:
- ``ListingKey``
- ``ListingContractDate``
- ``ListPrice``
- ``ClosePrice``
- ``PurchaseContractDate``
- ``CloseDate``
- ``LivingArea``
- ``BedroomsTotal``
- ``BathroomsTotalInteger``
- ``Latitude``/``Longitude``
- ``UnparsedAddress``

Since the above columns are not part of key fields, we can remove them.

In [146]:
# drop cols w >90% nulls
sold_clean = sold.drop(columns = flag_over_90)

print('Sold shape after dropping:', sold_clean.shape)
sold_clean.head()

Sold shape after dropping: (448253, 69)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,...,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgentAOR,ListAgentAOR,OriginatingSystemName,OriginatingSystemSubName,year_month,rate_30yr_fixed
0,"Carpet,Tile,Wood",True,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,Joan,Wachter,...,Other,94401,6472.0,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425
1,NaN,False,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,Michael,Darwich,...,NaN,91950,NaN,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425
2,NaN,False,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,Michael,Darwich,...,NaN,91950,NaN,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425
3,NaN,True,False,2100000.0,1067652762,jenniferarielkennedy@gmail.com,2024-01-02,2100000.0,Jennifer,Kennedy,...,San Luis Coastal Unified,93401,0.0,11219.0,NaN,NaN,NaN,NaN,2024-01,6.6425
4,Tile,True,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,Kira,Meade,...,NaN,95437,NaN,74487.6,NaN,NaN,NaN,NaN,2024-01,6.6425


### shape after dropping >90% null: (448253, 69)

We can also consider dropping columns with over 50% nulls for more meaningful analyses, but we will still make sure that none of these flagged columns are considered key data fields. From our Week 2-3 deliverable instructions, we were given a list of key numeric fields (in which I will define as ``core_fields``), so we should also make sure that they are not flagged.

In [147]:
# consider dropping cols w >50% nulls
flag_over_50 = sold_null_summary[sold_null_summary['null pct'] > 50].index.tolist()

# recall wk2-3: remove core fields from the list of cols to drop
core_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice',
               'LivingArea', 'LotSizeAcres', 'BedroomsTotal',
               'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

for field in core_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields)') # overlaps with >90% null cols
flag_over_50

27 columns with over 50% nulls (excl. core fields)


['AboveGradeFinishedArea',
 'AssociationFeeFrequency',
 'BasementYN',
 'BelowGradeFinishedArea',
 'BuilderName',
 'BuildingAreaTotal',
 'BusinessType',
 'BuyerAgencyCompensation',
 'BuyerAgencyCompensationType',
 'CoBuyerAgentFirstName',
 'CoListAgentFirstName',
 'CoListAgentLastName',
 'CoListOfficeName',
 'CoveredSpaces',
 'ElementarySchool',
 'ElementarySchoolDistrict',
 'FireplacesTotal',
 'HighSchool',
 'LotSizeDimensions',
 'MiddleOrJuniorSchool',
 'MiddleOrJuniorSchoolDistrict',
 'OriginatingSystemName',
 'OriginatingSystemSubName',
 'SubdivisionName',
 'TaxAnnualAmount',
 'TaxYear',
 'WaterfrontYN']

In week 6, we will be feature engineering using existing columns and also adding school districts using properties' latitude and longitude values, so we will exclude removing schools and school districts in case we end up populating them in future deliverables.

In [148]:
# remove schools and school districts from flagged cols
school_fields = ['ElementarySchool',
                 'ElementarySchoolDistrict',
                 'MiddleOrJuniorSchool',
                 'MiddleOrJuniorSchoolDistrict',
                 'HighSchool']

for field in school_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

for i in flag_over_90:
    for j in flag_over_50:
        if j in flag_over_90:
            flag_over_50.remove(j)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields & schools):')
flag_over_50

9 columns with over 50% nulls (excl. core fields & schools):


['AssociationFeeFrequency',
 'BuyerAgencyCompensation',
 'BuyerAgencyCompensationType',
 'CoListAgentFirstName',
 'CoListAgentLastName',
 'CoListOfficeName',
 'OriginatingSystemName',
 'OriginatingSystemSubName',
 'SubdivisionName']

In [149]:
# drop cols w >50% nulls (excl. core fields and schools)
sold_clean = sold_clean.drop(columns = flag_over_50)
print('Sold shape after dropping:', sold_clean.shape)

sold_clean.head()

Sold shape after dropping: (448253, 60)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgentAOR,ListAgentAOR,year_month,rate_30yr_fixed
0,"Carpet,Tile,Wood",True,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,Joan,Wachter,...,False,1.0,Other,94401,6472.0,NaN,NaN,NaN,2024-01,6.6425
1,NaN,False,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,Michael,Darwich,...,False,2.0,NaN,91950,NaN,NaN,NaN,NaN,2024-01,6.6425
2,NaN,False,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,Michael,Darwich,...,False,2.0,NaN,91950,NaN,NaN,NaN,NaN,2024-01,6.6425
3,NaN,True,False,2100000.0,1067652762,jenniferarielkennedy@gmail.com,2024-01-02,2100000.0,Jennifer,Kennedy,...,False,3.0,San Luis Coastal Unified,93401,0.0,11219.0,NaN,NaN,2024-01,6.6425
4,Tile,True,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,Kira,Meade,...,NaN,2.0,NaN,95437,NaN,74487.6,NaN,NaN,2024-01,6.6425


### shape after dropping >50% null: (448253, 60)

We can take a look at the remaining columns and determine what else to remove.

In [150]:
sorted(sold_clean.columns)

['AssociationFee',
 'AttachedGarageYN',
 'BathroomsTotalInteger',
 'BedroomsTotal',
 'BuyerAgentAOR',
 'BuyerAgentFirstName',
 'BuyerAgentLastName',
 'BuyerAgentMlsId',
 'BuyerOfficeAOR',
 'BuyerOfficeName',
 'City',
 'CloseDate',
 'ClosePrice',
 'ContractStatusChangeDate',
 'CountyOrParish',
 'DaysOnMarket',
 'ElementarySchool',
 'FireplaceYN',
 'Flooring',
 'GarageSpaces',
 'HighSchool',
 'HighSchoolDistrict',
 'Latitude',
 'Levels',
 'ListAgentAOR',
 'ListAgentEmail',
 'ListAgentFirstName',
 'ListAgentFullName',
 'ListAgentLastName',
 'ListOfficeName',
 'ListPrice',
 'ListingContractDate',
 'ListingId',
 'ListingKey',
 'ListingKeyNumeric',
 'LivingArea',
 'Longitude',
 'LotSizeAcres',
 'LotSizeArea',
 'LotSizeSquareFeet',
 'MLSAreaMajor',
 'MainLevelBedrooms',
 'MiddleOrJuniorSchool',
 'MlsStatus',
 'NewConstructionYN',
 'OriginalListPrice',
 'ParkingTotal',
 'PoolPrivateYN',
 'PostalCode',
 'PropertySubType',
 'PropertyType',
 'PurchaseContractDate',
 'StateOrProvince',
 'Stories',

In [151]:
# as we investigate the remaining columns, we can set up a list and add onto it
cols_to_remove = []

sold[['ListAgentFirstName', 'ListAgentFullName', 'ListAgentLastName', 'ListAgentEmail']].isna().sum()

ListAgentFirstName     3318
ListAgentFullName        95
ListAgentLastName        42
ListAgentEmail        79675
dtype: int64

In [152]:
sold['ListAgentFullName'].value_counts()

# we can remove list agent first name and last name since there's already a fullname col
cols_to_remove.append('ListAgentFirstName')
cols_to_remove.append('ListAgentLastName')

# not needed for analysis
cols_to_remove.append('ListAgentEmail')

In [153]:
# street numbers are unique + not needed for analysis
print(sold['StreetNumberNumeric'].value_counts())

cols_to_remove.append('StreetNumberNumeric')

StreetNumberNumeric
1.0        544
10.0       463
15.0       462
6.0        454
5.0        434
          ... 
49281.0      1
69657.0      1
72631.0      1
12879.0      1
80142.0      1
Name: count, Length: 48696, dtype: int64


In [154]:
sold[['ListingId', 'ListingKey', 'ListingKeyNumeric']].value_counts()

'''
ListingId   ListingKey  ListingKeyNumeric
SR25229893  1137564166  1137564166           3
CV25004341  1101502229  1101502229           3
OR24107765  1075843364  1075843364           3
MB24118151  1076326017  1076326017           3
OR24078258  1069697643  1069697643           2
                                            ..
CV25099944  1112966360  1112966360           1
CV25099932  1112687525  1112687525           1
CV25099884  1112680216  1112680216           1
CV25099848  1112869126  1112869126           1
WS26138647  1176037656  1176037656           1
Name: count, Length: 447914, dtype: int64
'''
sold_clean['ListingKeyNumeric'].equals(sold_clean['ListingKey'])
# True

cols_to_remove.append('ListingKeyNumeric') # since values are the same as listingkey

In [155]:
print(sold_clean[['PropertyType', 'PropertySubType']].value_counts())

cols_to_remove.append('PropertyType')

PropertyType  PropertySubType      
Residential   SingleFamilyResidence    335770
              Condominium               73655
              Townhouse                 26267
              ManufacturedOnLand         5795
              Duplex                     2491
              StockCooperative           1765
              Cabin                       508
              Triplex                     380
              MixedUse                    223
              Quadruplex                  159
              BoatSlip                     93
              OwnYourOwn                   66
              MobileHome                   65
              ManufacturedHome             52
              Loft                         31
              CoOwnership                  20
              Timeshare                    18
              Farm                         16
              Studio                       11
              DeededParking                 6
Name: count, dtype: int64


In [156]:
print(sold_clean[['LotSizeAcres', 'LotSizeArea', 'LotSizeSquareFeet']])

'''
        LotSizeAcres  LotSizeArea  LotSizeSquareFeet
0                NaN          NaN                NaN
1                NaN          NaN                NaN
2                NaN          NaN                NaN
3             0.2576     11219.00            11219.0
4             1.7100         1.71            74487.6    # area rep in acres
...              ...          ...                ...
448248        0.1631      7103.00             7103.0    # sq.ft
448249        0.6939     30228.00            30228.0
448250        0.1008      4389.00             4389.0    # area rep in sq.ft
448251        2.6124    113797.00           113797.0
448252     2452.6300      2452.63        106836562.8    # acres

[448253 rows x 3 columns]
'''

# units are not consistent in this column, values are either represented in acres or in square feet
cols_to_remove.append('LotSizeArea')

        LotSizeAcres  LotSizeArea  LotSizeSquareFeet
0                NaN          NaN                NaN
1                NaN          NaN                NaN
2                NaN          NaN                NaN
3             0.2576     11219.00            11219.0
4             1.7100         1.71            74487.6
...              ...          ...                ...
448248        0.1631      7103.00             7103.0
448249        0.6939     30228.00            30228.0
448250        0.1008      4389.00             4389.0
448251        2.6124    113797.00           113797.0
448252     2452.6300      2452.63        106836562.8

[448253 rows x 3 columns]


In [157]:
sold_clean['StateOrProvince'].value_counts()

StateOrProvince
CA    448223
AZ        10
OS         4
NV         2
CO         2
FL         2
ID         1
TX         1
GA         1
ME         1
NY         1
MO         1
TN         1
AR         1
VA         1
Name: count, dtype: int64

In [158]:
sold_clean['StateOrProvince'].unique()

array(['CA', 'NV', 'AZ', 'ID', 'TX', 'OS', 'GA', 'ME', 'NY', 'CO', 'MO',
       'TN', 'FL', 'AR', 'VA', nan], dtype=object)

In [159]:
# investigate nan value
sold_clean[sold_clean['StateOrProvince'].isnull()][['Latitude', 'Longitude', 'StateOrProvince']]

,Latitude,Longitude,StateOrProvince
435015,34.006183,-118.174188,NaN


In [160]:
''' 
cali coords
lat: (32.0, 42.5)
lon: (-125.0, -113.5)
'''
# since there's only 1 nan value, we can fill it with cali coords since lat/lon are within cali
sold_clean['StateOrProvince'] = sold_clean['StateOrProvince'].fillna('CA')
print(sold_clean.iloc[435015]['StateOrProvince'])

# double check that the nan value is gone
print('Double check:', sold_clean['StateOrProvince'].unique())

# since all properties are supposedly in cali, we dont need this column
cols_to_remove.append('StateOrProvince')

CA
Double check: ['CA' 'NV' 'AZ' 'ID' 'TX' 'OS' 'GA' 'ME' 'NY' 'CO' 'MO' 'TN' 'FL' 'AR'
 'VA']


In [161]:
sold_clean[['MLSAreaMajor', 'MlsStatus', 'BuyerAgentMlsId']]

# sold_clean['MlsStatus'].value_counts()
'''
MlsStatus
Closed    443797
Name: count, dtype: int64
'''

'\nMlsStatus\nClosed    443797\nName: count, dtype: int64\n'

In [162]:
# we remove these since BuyerAgentMlsId exists & we also wont be looking at the buyer's name
cols_to_remove.append('BuyerAgentFirstName')
cols_to_remove.append('BuyerAgentLastName')

### REMOVE COLUMNS

In [163]:
# double check before removing
cols_to_remove

['ListAgentFirstName',
 'ListAgentLastName',
 'ListAgentEmail',
 'StreetNumberNumeric',
 'ListingKeyNumeric',
 'PropertyType',
 'LotSizeArea',
 'StateOrProvince',
 'BuyerAgentFirstName',
 'BuyerAgentLastName']

#### reviewing columns to remove

| column name | description |
| ----------- | ---------- |
| ListAgentFirstName | The first name of the listing agent |
| ListAgentLastName | The last name of the listing agent |
| StreetNumberNumeric | The integer portion of the street number |
| PropertyType | The list of types of properties such as Residential, Lease, Income, Land, Mobile, Commercial Sale, etc |
| ListAgentEmail | The email address of the Listing Agent |
| LotSizeArea | The total area of the lot. See Lot Size Units for the units of measurement (Square Feet, Square Meters, Acres, etc) |
| StateOrProvince | Text field containing the accepted postal abbreviation for the state or province | 
| ListingKeyNumeric | same as ``ListingKey`` column |
| BuyerAgentFirstName | The first name of the buyer's agent |
| BuyerAgentLastName | The last name of the buyer's agent |

In [164]:
# drop columns
sold_clean = sold_clean.drop(columns = cols_to_remove)

In [165]:
print(sold_clean.shape)
print(sold_clean.columns)

(448253, 50)
Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice',
       'ListingKey', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude',
       'UnparsedAddress', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'ListAgentFullName',
       'BuyerAgentMlsId', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus',
       'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'BuyerOfficeAOR', 'YearBuilt',
       'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories',
       'HighSchool', 'Levels', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'BuyerAgentAOR', 'ListAgentAOR', 'year_month',
       'rate_30yr_fixed'],
      dtype='object')


### final shape after dropping: (448253, 50)

### CLEAN ROWS

now that we've dealt with columns, lets look into invalid values for ClosePrice, LivingArea, DOM, and negative bedrooms or bathrooms

In [166]:
# flag for negative closeprice
sold_clean['neg_closeprice_flag'] = sold_clean['ClosePrice'] <= 0

sold_clean[sold_clean['neg_closeprice_flag'] == True]

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgentAOR,ListAgentAOR,year_month,rate_30yr_fixed,neg_closeprice_flag
273605,"Carpet,Laminate",True,False,1350000.0,1117019091,2025-07-30,0.0,34.096368,-118.039289,5026 Doreen Avenue,...,2.0,NaN,91780,NaN,11313.0,PasadenaFoothills,PasadenaFoothills,2025-07,6.72,True


In [167]:
# flag for negative livingarea
sold_clean['neg_livingarea_flag'] = sold_clean['LivingArea'] <= 0

sold_clean[sold_clean['neg_livingarea_flag'] == True]

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgentAOR,ListAgentAOR,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag
7702,Wood,True,NaN,8295000.0,1046380894,2024-01-22,8000000.0,34.092696,-118.411241,1114 Sutton Way,...,NaN,90210,NaN,19987.0,NaN,NaN,2024-01,6.6425,False,True
21090,NaN,True,NaN,9998000.0,1046973939,2024-02-09,9000000.0,34.043167,-118.623635,20737 Cool Oak Way,...,NaN,90265,NaN,133305.0,NaN,NaN,2024-02,6.7760,False,True
22959,Wood,True,False,1950000.0,1045698780,2024-02-23,1930000.0,34.087410,-118.279744,1323 Edgecliffe Drive,...,NaN,90026,0.0,7349.0,NaN,NaN,2024-02,6.7760,False,True
22960,Wood,True,False,1995000.0,1045698270,2024-02-23,1850000.0,34.087403,-118.279964,1321 Edgecliffe Drive,...,NaN,90026,0.0,7349.0,NaN,NaN,2024-02,6.7760,False,True
35334,Wood,False,False,1790000.0,1054697515,2024-03-21,1730000.0,34.420540,-119.709457,313 W Victoria Street,...,NaN,93101,NaN,7841.0,NaN,NaN,2024-03,6.8200,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445915,NaN,False,False,750000.0,1153246696,2026-06-05,700000.0,33.785121,-117.808435,4223 E Marmon Avenue,...,NaN,92869,NaN,6470.0,Claw,Claw,2026-06,6.4900,False,True
446381,Concrete,False,False,89900.0,1152540534,2026-06-30,57000.0,33.701933,-116.262408,80394 Ave 48 101,...,NaN,92201,670.0,2178.0,CaliforniaDesert,CaliforniaDesert,2026-06,6.4900,False,True
447215,"Concrete,Tile",True,False,169000.0,1150495367,2026-06-26,160000.0,33.705581,-116.260428,80394 Avenue 48 281,...,NaN,92201,670.0,2178.0,CaliforniaDesert,CaliforniaDesert,2026-06,6.4900,False,True
447470,"Tile,Wood",True,False,699000.0,1147474972,2026-06-05,640000.0,NaN,NaN,661 Peralta Ave Unit 3,...,NaN,94110,400.0,2622.0,NaN,Oakland,2026-06,6.4900,False,True


In [168]:
# look @ DOM
sold_clean[['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'DaysOnMarket']].head()

,ListingContractDate,PurchaseContractDate,CloseDate,DaysOnMarket
0,2021-10-06,2023-11-22,2024-01-26,777
1,2021-03-08,2021-06-30,2024-01-05,33
2,2021-03-08,2021-11-18,2024-01-05,228
3,2023-11-15,2023-11-15,2024-01-02,0
4,2023-11-17,2023-12-19,2024-01-22,-36


In [169]:
# flag @ negative DOM
sold_clean['neg_dom_flag'] = sold_clean['DaysOnMarket'] < 0

sold_clean[sold_clean['neg_dom_flag'] == True].shape

(51, 53)

In [170]:
# look @ negative bedrooms
print(sold_clean[sold_clean['BedroomsTotal'] < 0].shape)
sold_clean[sold_clean['BedroomsTotal'] < 0].head()

(0, 53)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgentAOR,ListAgentAOR,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag


In [171]:
# look @ negative bathrooms
sold_clean[sold_clean['BathroomsTotalInteger'] < 0]

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgentAOR,ListAgentAOR,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag


### data consistency checks
- validate logical order of date fields (ListingContractDate < PurchaseContractDate < CloseDate)
- create boolean flag cols to mark records that violate these rules
    - listing_after_close_flag
    - purchase_after_close_flag
    - negative_timeline_flag

In [172]:
# validate order of datefields
invalid_rows = sold_clean[~((sold_clean['ListingContractDate'] < sold_clean['PurchaseContractDate']) & 
                      (sold_clean['PurchaseContractDate'] < sold_clean['CloseDate']))]

print('Shape of rows where date fields are out of order', invalid_rows.shape)
invalid_rows[['ListingContractDate', 'PurchaseContractDate', 'CloseDate']].head()

Shape of rows where date fields are out of order (14966, 53)


,ListingContractDate,PurchaseContractDate,CloseDate
3,2023-11-15,2023-11-15,2024-01-02
5,2024-01-31,2024-03-04,2024-01-31
9,2024-01-31,2024-02-05,2024-01-31
10,2024-01-29,2024-02-04,2024-01-29
12,2024-01-10,2024-01-10,2024-01-31


In [173]:
# create bool flag cols to mark records that violate these rules
# correct order: list date < purchase date < close date

# list date after close date
sold_clean['listing_after_close_flag'] = sold_clean['ListingContractDate'] > sold_clean['CloseDate']
print('# rows where list date is after close date:', sold_clean[sold_clean['listing_after_close_flag'] == True].shape[0])

# purchase date after close date
sold_clean['purchase_after_close_flag'] = sold_clean['PurchaseContractDate'] > sold_clean['CloseDate']
print('# rows where purchase date is after close date:', sold_clean[sold_clean['purchase_after_close_flag'] == True].shape[0])

# violates order
sold_clean['negative_timeline_flag'] = ~((sold_clean['ListingContractDate'] < sold_clean['PurchaseContractDate']) & 
                                         (sold_clean['PurchaseContractDate'] < sold_clean['CloseDate']))
print('# rows with illogical timeline:', sold_clean[sold_clean['negative_timeline_flag'] == True].shape[0])

# check that these columns were made
sold_clean[['listing_after_close_flag', 'purchase_after_close_flag', 'negative_timeline_flag']].head()

# rows where list date is after close date: 70
# rows where purchase date is after close date: 239
# rows with illogical timeline: 14966


,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,True
4,False,False,False


### geographic data checks
- flag records w missing coords (lat/lon is null)
- flag lat = 0 or lon = 0 (sentinel null vals)
- flag lon > 0 errors (cal coords should be negative)
- flag out-of-state/implausible coords

In [174]:
# flag missing coords (lat/lon is null)
sold_clean['missing_coords_flag'] = (sold_clean['Latitude'].isna()) | (sold_clean['Longitude'].isna())

print(sold_clean[sold_clean['missing_coords_flag'] == True].shape)
sold_clean[sold_clean['missing_coords_flag'] == True].head()

(4388, 57)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,ListAgentAOR,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag
448,"Carpet,Tile",NaN,False,1325000.0,1054150881,2024-01-30,1335000.0,NaN,NaN,NaN,...,NaN,2024-01,6.6425,False,False,False,False,False,True,True
671,NaN,False,False,624900.0,1054077195,2024-01-23,640000.0,NaN,NaN,8378 New Salem #19,...,NaN,2024-01,6.6425,False,False,False,False,False,False,True
855,"Carpet,Tile",NaN,False,1110000.0,1054007898,2024-01-11,1025000.0,NaN,NaN,NaN,...,NaN,2024-01,6.6425,False,False,False,False,False,True,True
3566,Laminate,NaN,False,495000.0,1050249809,2024-01-04,478000.0,NaN,NaN,1450 Thrush Ave #14,...,NaN,2024-01,6.6425,False,False,False,False,False,False,True
3726,"Carpet,Tile,Wood",NaN,False,2399000.0,1049915635,2024-01-24,2326000.0,NaN,NaN,NaN,...,NaN,2024-01,6.6425,False,False,False,False,False,True,True


In [175]:
# lat = 0 or lon = 0
sold_clean['sentinel_coords_flag'] = (sold_clean['Latitude'] == 0) | (sold_clean['Longitude'] == 0)

print(sold_clean[sold_clean['sentinel_coords_flag'] == True].shape)
sold_clean[sold_clean['sentinel_coords_flag'] == True].head()

(37, 58)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag
138003,"Carpet,Tile",False,True,449888.0,1076725468,2024-09-12,420000.0,0.0,0.0,44526 Sancroft Avenue,...,2024-09,6.180,False,False,False,False,False,False,False,True
141073,"Carpet,Vinyl",True,False,399000.0,1064843666,2024-09-06,360000.0,0.0,0.0,25604 6th Street,...,2024-09,6.180,False,False,False,False,False,False,False,True
157184,"Carpet,Tile,Vinyl",NaN,False,955515.0,1072840823,2024-10-16,1060487.0,0.0,0.0,3414 Rywood Street,...,2024-10,6.428,False,False,False,False,False,False,False,True
158237,NaN,False,NaN,872008.0,1093509478,2024-11-22,844000.0,0.0,0.0,1960 Marigold St.,...,2024-11,6.805,False,False,False,False,False,False,False,True
180171,"Carpet,Tile,Wood",False,NaN,1598000.0,1089715991,2024-12-23,1528000.0,0.0,0.0,706 Fin Court,...,2024-12,6.715,False,False,False,False,False,False,False,True


In [176]:
# lon > 0 errors (cali coords should be negative)
sold_clean['pos_lon_flag'] = sold_clean['Longitude'] > 0

print(sold_clean[sold_clean['pos_lon_flag'] == True].shape)
sold_clean[sold_clean['pos_lon_flag'] == True].head()

(31, 59)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag
9714,NaN,False,False,810000.0,1044978778,2024-01-23,735000.0,1.000000,2.000000,1202 W Pine Street,...,6.6425,False,False,False,False,False,False,False,False,True
23077,NaN,False,False,798000.0,1045537743,2024-02-02,730000.0,33.934625,117.404219,3374 Madison,...,6.7760,False,False,False,False,False,False,False,False,True
37266,NaN,True,False,709655.0,1050732402,2024-03-04,710080.0,32.726310,116.973758,2014 Adamite Street,...,6.8200,False,False,False,False,False,False,False,False,True
62742,NaN,True,False,315000.0,1068094062,2024-05-09,315000.0,50.000000,100.000000,1999 Highland Drive,...,7.0600,False,False,False,False,False,True,False,False,True
65556,NaN,True,False,425000.0,1065274412,2024-05-13,425000.0,34.475681,117.372126,13346 Crystal Court,...,7.0600,False,False,False,False,False,False,False,False,True


In [177]:
# out of state (oos) or implausible coords

'''cali coords range
lat: (32.0, 42.5)
lon: (-125.0, -113.5)
'''

# filter for only cali listings
sold_clean['oos_coords_flag'] = ~(sold_clean['Latitude'].between(32.0, 42.5) & sold_clean['Longitude'].between(-125.0, -113.5))

sold_clean['oos_coords_flag'].value_counts()

oos_coords_flag
False    443765
True       4488
Name: count, dtype: int64

In [178]:
print('# rows with missing coordinates:', sold_clean[sold_clean['missing_coords_flag'] == True].shape[0])
print('# rows with sentinel coordinates:', sold_clean[sold_clean['sentinel_coords_flag'] == True].shape[0])

print('# rows with non-California values (negative longitude):', sold_clean[sold_clean['pos_lon_flag'] == True].shape[0])
print('# rows with out of state/implausible coordinates:', sold_clean[sold_clean['oos_coords_flag'] == True].shape[0])

# rows with missing coordinates: 4388
# rows with sentinel coordinates: 37
# rows with non-California values (negative longitude): 31
# rows with out of state/implausible coordinates: 4488


### review flagged cols

We created about 10 flagged columns:
- neg_closeprice_flag
- neg_livingarea_flag
- neg_dom_flag
- listing_after_close_flag
- purchase_after_close_flag
- negative_timeline_flag
- missing_coords_flag
- sentinel_coords_flag
- pos_lon_flag
- oos_coords_flag

So now we can start cleaning by rows

In [179]:
sold_clean.iloc[:,-10:]

,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag
0,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,True,False,False,False,False
4,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
448248,False,False,False,False,False,False,False,False,False,False
448249,False,False,False,False,False,False,False,False,False,False
448250,False,False,False,False,False,False,False,False,False,False
448251,False,False,False,False,False,False,False,False,False,False


In [180]:
sold_clean[sold_clean['neg_closeprice_flag'] == True]

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag
273605,"Carpet,Laminate",True,False,1350000.0,1117019091,2025-07-30,0.0,34.096368,-118.039289,5026 Doreen Avenue,...,True,False,False,False,False,False,False,False,False,False


In [181]:
# remove the singular 0-value closeprice
sold_clean = sold_clean[sold_clean['neg_closeprice_flag'] == False]

print(sold_clean.shape)
sold_clean.head(3)

(448252, 60)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag
0,"Carpet,Tile,Wood",True,False,499000.0,551985747,2024-01-26,240000.0,37.566106,-122.327954,1 Baldwin Avenue 411,...,False,False,False,False,False,False,False,False,False,False
1,NaN,False,False,759900.0,522107581,2024-01-05,815000.0,32.659315,-117.096922,2811 C Avenue,...,False,False,False,False,False,False,False,False,False,False
2,NaN,False,False,739900.0,510919001,2024-01-05,810000.0,32.659284,-117.097174,2812 C Avenue,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# remove 0-value livingarea
sold_clean[sold_clean['neg_livingarea_flag'] == True]['LivingArea'].value_counts()

# LivingArea
# 0.0    165
# Name: count, dtype: int64

# remove 0-value livingarea
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[sold_clean['neg_livingarea_flag'] == False]

print('Shape after removing:', sold_clean.shape)
sold_clean.head(3)

(448087, 60)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag
0,"Carpet,Tile,Wood",True,False,499000.0,551985747,2024-01-26,240000.0,37.566106,-122.327954,1 Baldwin Avenue 411,...,False,False,False,False,False,False,False,False,False,False
1,NaN,False,False,759900.0,522107581,2024-01-05,815000.0,32.659315,-117.096922,2811 C Avenue,...,False,False,False,False,False,False,False,False,False,False
2,NaN,False,False,739900.0,510919001,2024-01-05,810000.0,32.659284,-117.097174,2812 C Avenue,...,False,False,False,False,False,False,False,False,False,False


In [183]:
# fix neg days on market values
mask = sold_clean[sold_clean['neg_dom_flag'] == True]

print('# of rows with negative days on market:', mask[['ListingContractDate', 'CloseDate', 'DaysOnMarket']].shape[0])
mask[['ListingContractDate', 'CloseDate', 'DaysOnMarket']].head()

# of rows with negative days on market: 51


,ListingContractDate,CloseDate,DaysOnMarket
4,2023-11-17,2024-01-22,-36
393,2024-01-08,2024-01-19,-1
11205,2024-02-24,2024-02-26,-13
11372,2024-02-16,2024-02-23,-10
18689,2023-12-12,2024-02-05,-10


In [184]:
# attempt to fix negative dom
sold_clean.loc[sold_clean['neg_dom_flag'] == True, 'DaysOnMarket'] = (sold_clean['CloseDate'] - sold_clean['ListingContractDate']).dt.days

mask[['ListingContractDate', 'CloseDate', 'DaysOnMarket']].head(3)

,ListingContractDate,CloseDate,DaysOnMarket
4,2023-11-17,2024-01-22,-36
393,2024-01-08,2024-01-19,-1
11205,2024-02-24,2024-02-26,-13


In [ ]:
# even after attempting to fix the negative dom, the values remain unchanged, so this could be
# because close dates were inputted before listing dates, so we can set these 51 entries to nulls
# since it makes up a small amount of our data
print('# rows with negative DOM:', sold_clean[sold_clean['neg_dom_flag'] == True]['DaysOnMarket'].shape)

print('# nulls before conversion:', sold_clean['DaysOnMarket'].isna().sum())
sold_clean.loc[sold_clean['neg_dom_flag'] == True, 'DaysOnMarket'] = np.nan

# check that negatives were converted to nulls
print('# nulls:', sold_clean['DaysOnMarket'].isna().sum())

# previous attempt: remove negative dom rows
# sold_clean = sold_clean[sold_clean['neg_dom_flag'] == False]

print('# nulls after conversion:', sold_clean['DaysOnMarket'].isna().sum())

Shape before converting negative to null: (448087, 60)
51
Shape after converting negative to null: (448087, 60)


In [186]:
print('# of rows where listing date after close date:', sold_clean[sold_clean['listing_after_close_flag'] == True].shape[0])
sold_clean[sold_clean['listing_after_close_flag'] == True][['ListingContractDate', 'CloseDate', 'DaysOnMarket']]

# of rows where listing date after close date: 70


,ListingContractDate,CloseDate,DaysOnMarket
20,2024-01-31,2024-01-30,0.0
81,2024-01-26,2024-01-25,0.0
710,2024-01-02,2024-01-01,0.0
24270,2024-04-08,2024-03-29,0.0
24390,2024-03-21,2024-03-20,0.0
...,...,...,...
414582,2026-05-18,2026-05-06,0.0
414813,2026-05-08,2026-05-07,0.0
430809,2026-06-25,2026-06-23,0.0
430860,2026-06-22,2026-06-17,0.0


In [187]:
sold_clean[(sold_clean['listing_after_close_flag'] == True) & (sold_clean['DaysOnMarket'] > 0)][['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'DaysOnMarket']].head(3)

,ListingContractDate,PurchaseContractDate,CloseDate,DaysOnMarket
129044,2024-10-02,2024-09-03,2024-09-19,16.0
129234,2024-09-30,2024-08-20,2024-09-05,16.0
129868,2024-09-28,2024-09-05,2024-09-24,20.0


In [188]:
# idea: use days on market to calculate the correct listing date
# actually i cant bc some are straight up out of order so maybe i should just null these rows?

# convert listing dates after close date into null
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['listing_after_close_flag'] == True, ['ListingContractDate', 'CloseDate']] = np.nan

# sold_clean = sold_clean[sold_clean['listing_after_close_flag'] == False]

print('Shape after conversion:', sold_clean.shape)
print(sold_clean[['ListingContractDate', 'CloseDate']].isna().sum())

Shape before conversion: (448087, 60)
Shape after conversion: (448087, 60)
ListingContractDate    71
CloseDate              70
dtype: int64


In [189]:
# investigate purchase date after close date
mask = sold_clean[sold_clean['purchase_after_close_flag'] == True]
print(mask.shape)

mask[['ListingContractDate', 'CloseDate', 'PurchaseContractDate']].head(3)

(239, 60)


,ListingContractDate,CloseDate,PurchaseContractDate
5,2024-01-31,2024-01-31,2024-03-04
9,2024-01-31,2024-01-31,2024-02-05
10,2024-01-29,2024-01-29,2024-02-04


In [190]:
# there isnt really a way to correct these so they can be converted to null
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['purchase_after_close_flag'] == True, ['ListingContractDate', 'CloseDate']] = np.nan

print('Shape after conversion:', sold_clean.shape)
print(sold_clean[['ListingContractDate', 'CloseDate']].isna().sum())

Shape before conversion: (448087, 60)
Shape after conversion: (448087, 60)
ListingContractDate    308
CloseDate              307
dtype: int64


In [191]:
# investigate negative timeline flag
mask = sold_clean[sold_clean['negative_timeline_flag'] == True]
print(mask.shape)

mask[['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'DaysOnMarket']].head(3)

(14940, 60)


,ListingContractDate,PurchaseContractDate,CloseDate,DaysOnMarket
3,2023-11-15,2023-11-15,2024-01-02,0.0
5,NaT,2024-03-04,NaT,0.0
9,NaT,2024-02-05,NaT,0.0


In [192]:
# as before, there isnt a way to fix the wrong dates, so we can turn them into null
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['negative_timeline_flag'] == True, ['ListingContractDate', 'CloseDate']] = np.nan

print('Shape after conversion:', sold_clean.shape)
print(sold_clean[['ListingContractDate', 'CloseDate']].isna().sum())


Shape before conversion: (448087, 60)
Shape after conversion: (448087, 60)
ListingContractDate    14940
CloseDate              14940
dtype: int64


In [193]:
# investigate missing coords flag
mask = sold_clean[sold_clean['missing_coords_flag'] == True]
print('# rows with missing coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

# rows with missing coordinates: 4385


,Latitude,Longitude,City,PostalCode
448,NaN,NaN,Newark,94560
671,NaN,NaN,San Diego,92126
855,NaN,NaN,Concord,94521
3566,NaN,NaN,San Leandro,94578
3726,NaN,NaN,Berkeley,94705


In [ ]:
# there isn't a way to correct lat/lon values but we can keep them
# since we can use other information in the row for analysis
# print('Shape before removing:', sold_clean.shape)
# sold_clean = sold_clean[sold_clean['missing_coords_flag'] == False]

# print('Shape after removing:', sold_clean.shape)

Shape before removing: (448087, 60)
Shape after removing: (443702, 60)


In [195]:
# investigate sentinel coords
mask = sold_clean[sold_clean['sentinel_coords_flag'] == True]
print('# rows with sentinel coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

# rows with sentinel coordinates: 37


,Latitude,Longitude,City,PostalCode
138003,0.0,0.0,Lancaster,93535
141073,0.0,0.0,San Bernardino,92410
157184,0.0,0.0,Manteca,95337
158237,0.0,0.0,Hollister,95023
180171,0.0,0.0,San Ramon,94583


In [196]:
# remove 0 lat/lon values
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[sold_clean['sentinel_coords_flag'] == False]

print('Shape after removing:', sold_clean.shape)

Shape before removing: (443702, 60)
Shape after removing: (443665, 60)


In [197]:
# investigate positive longitude flag
mask = sold_clean[sold_clean['pos_lon_flag'] == True]
print('# rows outside CA:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

# rows outside CA: 31


,Latitude,Longitude,City,PostalCode
9714,1.000000,2.000000,Santa Ana,92703
23077,33.934625,117.404219,Riverside,92504
37266,32.726310,116.973758,Spring Valley,91977
62742,50.000000,100.000000,Merced,95340
65556,34.475681,117.372126,Victorville,92392
67740,33.852530,118.251830,Carson,90746
74991,35.251598,120.634389,San Luis Obispo,93401
116897,33.940160,118.236160,Los Angeles,90002
120229,34.412880,117.276000,Hesperia,92345
144610,33.702242,117.939341,Fountain Valley,92708


In [ ]:
# set coordinate values to null
# even though cities correspond to their zipcodes, the lat/lon aren't within CA bounds
print('Shape before conversion:', sold_clean[sold_clean['pos_lon_flag'] == True].shape)
sold_clean.loc[sold_clean['pos_lon_flag'] == True, ['Latitude', 'Longitude']] = np.nan

print('Shape after conversion:', sold_clean[sold_clean['pos_lon_flag'] == True].shape)
sold_clean[sold_clean['pos_lon_flag'] == True][['Latitude', 'Longitude']].head()

Shape before conversion: (31, 60)
Shape after conversion: (31, 60)


,Latitude,Longitude
9714,NaN,NaN
23077,NaN,NaN
37266,NaN,NaN
62742,NaN,NaN
65556,NaN,NaN


In [199]:
# mask['City'].unique()

# array(['Santa Ana', 'Riverside', 'Spring Valley', 'Merced', 'Victorville',
#        'Carson', 'San Luis Obispo', 'Los Angeles', 'Hesperia',
#        'Fountain Valley', 'East Los Angeles', 'Orland', 'San Bernardino',
#        'Beaumont', 'Julian', 'Hawthorne', 'Palos Verdes Estates',
#        'Visalia', 'Fresno', 'Pico Rivera', 'Santa Maria', 'Lake Elsinore',
#        'Moreno Valley', 'Lancaster', 'Helendale'], dtype=object)



# mask['PostalCode'].unique()
# 
# array(['92703', '92504', '91977', '95340', '92392', '90746', '93401',
#        '90002', '92345', '92708', '90063', '95963', '92404', '90011',
#        '92395', '92223', '92036', '90250', '90274', '93291', '93730',
#        '90060', '93455', '92532', '92553', '92344', '93534', '92503',
#        '92342'], dtype=object)

In [200]:
# investigate out of state coords
mask = sold_clean[sold_clean['oos_coords_flag'] == True]
print('# rows with out of state/implausible coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

# rows with out of state/implausible coordinates: 63


,Latitude,Longitude,City,PostalCode
9714,NaN,NaN,Santa Ana,92703
23077,NaN,NaN,Riverside,92504
27960,48.428421,-123.365644,Sugarloaf,92386
37266,NaN,NaN,Spring Valley,91977
39822,47.675193,-116.786422,Outside Area (Outside Ca),83814
...,...,...,...,...
362095,NaN,NaN,Hesperia,92344
370579,NaN,NaN,Lancaster,93534
401765,NaN,NaN,Riverside,92503
402988,NaN,NaN,Helendale,92342


In [201]:
mask['City'].unique()

array(['Santa Ana', 'Riverside', 'Sugarloaf', 'Spring Valley',
       'Outside Area (Outside Ca)', 'Merced', 'Victorville', 'Carson',
       'San Luis Obispo', 'Carmel', 'Laguna Woods', 'Cobb', nan, 'Corona',
       'Los Angeles', 'Hesperia', 'Fountain Valley', 'East Los Angeles',
       'Orland', 'San Bernardino', 'Flagstaff',
       'Outside Area (Outside U.S.) Foreign Country', 'Palmdale', 'Other',
       'Beaumont', 'Monterey', 'Julian', 'Mountain House', 'Hawthorne',
       'Palos Verdes Estates', 'Visalia', 'Fresno', 'Pico Rivera',
       'Santa Maria', 'Lake Elsinore', 'Moreno Valley', 'Pomona',
       'Lancaster', 'Helendale', 'California City'], dtype=object)

In [202]:
# look into null cities
remove_mask = sold_clean[((sold_clean['oos_coords_flag'] == True) & (sold_clean['City'].isnull()))]
print(remove_mask.shape)
remove_mask

(5, 60)


,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag
102545,Wood,True,False,4700000.00,1075660480,2024-07-19,4600000.0,40.688808,-73.975784,29 S Elliott Place,...,False,False,False,False,False,False,False,False,False,True
134763,Wood,NaN,False,178000.00,1077710035,NaT,149000.0,38.598567,-90.354151,112 Oakwood Ave,...,False,False,False,False,False,True,False,False,False,True
254402,"Carpet,Tile,Wood",True,False,4500000.00,1093760214,2025-05-23,4050000.0,40.555581,-104.954622,2823 Majestic View Drive,...,False,False,False,False,False,False,False,False,False,True
270152,NaN,True,False,609448.57,1088831243,2025-06-05,558746.0,20.666287,-103.362745,1699 C. FermiN Riestra,...,False,False,False,False,False,False,False,False,False,True
301216,Tile,False,False,338528.00,1097409160,2025-08-06,299500.0,33.640831,-112.371086,14444 W Moccasin Trail,...,False,False,False,False,False,False,False,False,False,True


In [203]:
# remove non-CA rows
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[~((sold_clean['oos_coords_flag'] == True) & (sold_clean['City'].isnull()))]

print('Shape after removing:', sold_clean.shape)

Shape before removing: (443665, 60)
Shape after removing: (443660, 60)


In [204]:
# look into out of state
mask = sold_clean[(sold_clean['City'] == 'Outside Area (Outside Ca)') |
           (sold_clean['City'] == 'Other') |
           (sold_clean['City'] == 'Outside Area (Outside U.S.) Foreign Country')][['Latitude', 'Longitude', 'City', 'PostalCode']]
print(mask.shape)

mask

(30, 4)


,Latitude,Longitude,City,PostalCode
32071,35.189088,-114.006128,Outside Area (Outside Ca),86401
39822,47.675193,-116.786422,Outside Area (Outside Ca),83814
53800,29.794143,-95.149214,Outside Area (Outside Ca),77530
56232,33.658134,-112.393471,Outside Area (Outside Ca),85374
68664,34.984405,-85.282577,Outside Area (Outside Ca),30741
118385,38.478321,-107.876175,Outside Area (Outside Ca),81401
126120,32.662034,-114.484477,Outside Area (Outside Ca),85365
212108,36.404467,-82.356471,Outside Area (Outside Ca),37604
221789,31.853187,-116.613551,Outside Area (Outside U.S.) Foreign Country,22880
221792,31.868462,-116.645616,Outside Area (Outside Ca),22870


In [205]:
# remove out of state coords except cities marked 'Other'

print('Shape before cleaning:', sold_clean.shape)
sold_clean = sold_clean[
    ~sold_clean['City'].isin([
        'Outside Area (Outside Ca)',
        'Outside Area (Outside U.S.) Foreign Country'
    ])
]

print('Shape after cleaning:', sold_clean.shape)

Shape before cleaning: (443660, 60)
Shape after cleaning: (443648, 60)


In [206]:
# double check rows were removed
sold_clean[sold_clean['oos_coords_flag']]['City'].unique()

array(['Santa Ana', 'Riverside', 'Sugarloaf', 'Spring Valley', 'Merced',
       'Victorville', 'Carson', 'San Luis Obispo', 'Carmel',
       'Laguna Woods', 'Cobb', 'Corona', 'Los Angeles', 'Hesperia',
       'Fountain Valley', 'East Los Angeles', 'Orland', 'San Bernardino',
       'Flagstaff', 'Palmdale', 'Other', 'Beaumont', 'Monterey', 'Julian',
       'Mountain House', 'Hawthorne', 'Palos Verdes Estates', 'Visalia',
       'Fresno', 'Pico Rivera', 'Santa Maria', 'Lake Elsinore',
       'Moreno Valley', 'Pomona', 'Lancaster', 'Helendale',
       'California City'], dtype=object)

In [207]:
# check postal codes within cali
sold_clean[
    (sold_clean['City'] == 'Other') &
    (sold_clean['PostalCode'].str.startswith('9'))
    ][['City', 'PostalCode']]

,City,PostalCode
297274,Other,95493
300076,Other,90032
310889,Other,92109
363841,Other,90063
367910,Other,93603
370253,Other,95602
375979,Other,90044
378989,Other,90011
396323,Other,93654
413545,Other,90063


In [208]:
# rename cities correctly according to zipcode
zip_to_city = {
    95493: 'Witter Springs',
    90032: 'Los Angeles',
    92109: 'San Diego',
    90063: 'Los Angeles',
    93603: 'Badger',
    95602: 'Auburn',
    90044: 'Los Angeles',
    90011: 'Los Angeles',
    93654: 'Reedley'
}

mask = sold_clean['City'] == 'Other'

# apply fixes
sold_clean.loc[mask, 'City'] = (
    sold_clean.loc[mask, 'PostalCode']
    .map(zip_to_city)
)

In [209]:
print(sold_clean[sold_clean['oos_coords_flag']].shape)
sold_clean[sold_clean['oos_coords_flag']][['Latitude', 'Longitude', 'City', 'PostalCode']]

(50, 60)


,Latitude,Longitude,City,PostalCode
9714,NaN,NaN,Santa Ana,92703
23077,NaN,NaN,Riverside,92504
27960,48.428421,-123.365644,Sugarloaf,92386
37266,NaN,NaN,Spring Valley,91977
62742,NaN,NaN,Merced,95340
65556,NaN,NaN,Victorville,92392
67740,NaN,NaN,Carson,90746
74991,NaN,NaN,San Luis Obispo,93401
75166,10.305220,-84.810691,Carmel,93923
88675,-38.718318,-62.266348,Laguna Woods,92637


In [210]:
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['oos_coords_flag'] == True, ['Latitude', 'Longitude']] = np.nan

print('Shape after conversion:', sold_clean.shape)
sold_clean[sold_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']]

Shape before conversion: (443648, 60)
Shape after conversion: (443648, 60)


,Latitude,Longitude,City,PostalCode
9714,NaN,NaN,Santa Ana,92703
23077,NaN,NaN,Riverside,92504
27960,NaN,NaN,Sugarloaf,92386
37266,NaN,NaN,Spring Valley,91977
62742,NaN,NaN,Merced,95340
65556,NaN,NaN,Victorville,92392
67740,NaN,NaN,Carson,90746
74991,NaN,NaN,San Luis Obispo,93401
75166,NaN,NaN,Carmel,93923
88675,NaN,NaN,Laguna Woods,92637


In [211]:
# look into nulls and flagstaff area
sold_clean[(sold_clean['oos_coords_flag'] == True) &
           ((sold_clean['City'].isna()) |
           (sold_clean['City'] == 'Flagstaff'))
           ][['City', 'PostalCode']]

,City,PostalCode
196581,Flagstaff,88888
267289,NaN,32909
288368,NaN,33308
326032,NaN,86314
332343,NaN,00000
359362,NaN,22785
359389,NaN,22880


In [212]:
# remove out of state rows + row with flagstaff since that's in AZ
mask = (
    sold_clean['oos_coords_flag'] &
    (sold_clean['City'].isna() |
     (sold_clean['City'] == 'Flagstaff'))
)

print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[~mask]

print('Shape after removing:', sold_clean.shape)

Shape before removing: (443648, 60)
Shape after removing: (443641, 60)


In [213]:
sold_clean[sold_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']]

,Latitude,Longitude,City,PostalCode
9714,NaN,NaN,Santa Ana,92703
23077,NaN,NaN,Riverside,92504
27960,NaN,NaN,Sugarloaf,92386
37266,NaN,NaN,Spring Valley,91977
62742,NaN,NaN,Merced,95340
65556,NaN,NaN,Victorville,92392
67740,NaN,NaN,Carson,90746
74991,NaN,NaN,San Luis Obispo,93401
75166,NaN,NaN,Carmel,93923
88675,NaN,NaN,Laguna Woods,92637


In [214]:
# fix palmdale zipcode
sold_clean.loc[319267, 'PostalCode'] = 93551 # used to be 83551

In [ ]:
# drop flagged columns that're done
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean.drop(columns = ['neg_closeprice_flag',
                                        'neg_livingarea_flag',
                                        'neg_dom_flag',
                                        'listing_after_close_flag',
                                        'purchase_after_close_flag',
                                        'negative_timeline_flag',
                                        'missing_coords_flag',
                                        'sentinel_coords_flag',
                                        'pos_lon_flag',
                                        'oos_coords_flag'])

print('Shape after removing:', sold_clean.shape)

Shape before removing: (443641, 60)
Shape after removing: (443641, 50)


In [216]:
# sold_clean.iloc[:,-10:].head(3)

In [217]:
sold_clean.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice',
       'ListingKey', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude',
       'UnparsedAddress', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'ListAgentFullName',
       'BuyerAgentMlsId', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus',
       'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'BuyerOfficeAOR', 'YearBuilt',
       'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories',
       'HighSchool', 'Levels', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'BuyerAgentAOR', 'ListAgentAOR', 'year_month',
       'rate_30yr_fixed'],
      dtype='object')

### deliverable
- document every xformation made & why
- include:
    - before/after counts
    - dtype confirmations
    - date consistency flag counts
    - geographic data quality summary noting invalid coordinate records